# Model Evaluation & Comparison
## Unsupervised Anomaly Detection — Performance Analysis

---

### Objective
เนื่องจากเราใช้ **Unsupervised Learning** (ไม่มี label ground truth) จึงไม่สามารถใช้ Accuracy/F1 ได้ตรงๆ

Notebook นี้จะประเมินผลจาก **4 มุม**:
1. **Internal Validation** — Silhouette Score, Cluster Quality
2. **Model Agreement** — Overlap & Consensus ระหว่าง 3 Models
3. **Anomaly Profile Analysis** — ธุรกรรมที่ถูก flag ต่างจากปกติอย่างไร
4. **Sensitivity & Stability** — Contamination threshold impact
5. **Business Insights** — สรุปสิ่งที่ค้นพบเชิง actionable

---

## 1. Setup & Reproduce Models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import LocalOutlierFactor
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

# Color palette
C_NORMAL = '#3b82f6'
C_ANOMALY = '#ef4444'
C_ISO = '#8b5cf6'
C_DBSCAN = '#f59e0b'
C_AE = '#10b981'
C_BG = '#f8fafc'

print('Libraries loaded')

In [ ]:
# Load raw data and reproduce feature engineering
df = pd.read_csv('bank_transactions.csv')
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], format='mixed')
print(f'Raw data: {df.shape[0]:,} rows x {df.shape[1]} columns')

# ── Feature Engineering (reproduce from Notebook 02) ──
# Account-level
acct = df.groupby('AccountID').agg(
    acct_txn_count=('TransactionID','count'),
    acct_mean_amount=('TransactionAmount','mean'),
    acct_std_amount=('TransactionAmount','std'),
    acct_max_amount=('TransactionAmount','max'),
    acct_median_amount=('TransactionAmount','median'),
    acct_mean_balance=('AccountBalance','mean'),
    acct_mean_duration=('TransactionDuration','mean'),
    acct_mean_login=('LoginAttempts','mean'),
    acct_unique_devices=('DeviceID','nunique'),
    acct_unique_locations=('Location','nunique'),
    acct_unique_merchants=('MerchantID','nunique'),
    acct_unique_channels=('Channel','nunique'),
    acct_unique_ips=('IP Address','nunique')
).reset_index()
acct['acct_std_amount'] = acct['acct_std_amount'].fillna(0)
df = df.merge(acct, on='AccountID', how='left')

# Transaction-level
df['amount_zscore'] = (df['TransactionAmount'] - df['acct_mean_amount']) / df['acct_std_amount'].replace(0,1)
df['amount_to_balance_ratio'] = df['TransactionAmount'] / df['AccountBalance'].replace(0,1)
df['amount_to_max_ratio'] = df['TransactionAmount'] / df['acct_max_amount'].replace(0,1)
df['duration_deviation'] = abs(df['TransactionDuration'] - df['acct_mean_duration'])
df['login_above_avg'] = (df['LoginAttempts'] > df['acct_mean_login']).astype(int)
df['high_login_flag'] = (df['LoginAttempts'] >= 3).astype(int)
Q1, Q3 = df['TransactionAmount'].quantile(0.25), df['TransactionAmount'].quantile(0.75)
df['amount_outlier_flag'] = (df['TransactionAmount'] > Q3 + 1.5*(Q3-Q1)).astype(int)

# Device & Location
dev = df.groupby('DeviceID')['AccountID'].nunique().reset_index(columns={'AccountID':'device_shared_accounts'})
dev.columns = ['DeviceID','device_shared_accounts']
df = df.merge(dev, on='DeviceID', how='left')
df['device_shared_flag'] = (df['device_shared_accounts'] > 1).astype(int)
ip_cnt = df.groupby('IP Address')['AccountID'].nunique().reset_index()
ip_cnt.columns = ['IP Address','ip_shared_accounts']
df = df.merge(ip_cnt, on='IP Address', how='left')
mf = df.groupby(['AccountID','MerchantID']).size().reset_index(name='merchant_visit_count')
df = df.merge(mf, on=['AccountID','MerchantID'], how='left')
df['is_new_merchant'] = (df['merchant_visit_count'] == 1).astype(int)

# Time-based
df['hour'] = df['TransactionDate'].dt.hour
df['day_of_week'] = df['TransactionDate'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['txn_date'] = df['TransactionDate'].dt.date
dv = df.groupby(['AccountID','txn_date']).size().reset_index(name='daily_txn_count')
df = df.merge(dv, on=['AccountID','txn_date'], how='left')
df = df.sort_values(['AccountID','TransactionDate'])
df['time_since_last_txn'] = df.groupby('AccountID')['TransactionDate'].diff().dt.total_seconds()/3600
df['time_since_last_txn'] = df['time_since_last_txn'].fillna(-1)
df['rapid_txn_flag'] = ((df['time_since_last_txn']>=0)&(df['time_since_last_txn']<1)).astype(int)
df.drop(columns=['txn_date'], inplace=True)

print(f'Features engineered. Total columns: {len(df.columns)}')

In [ ]:
# ── Prepare feature matrix & run all 3 models ──
features_for_model = [
    'TransactionAmount','TransactionDuration','LoginAttempts','AccountBalance',
    'amount_zscore','amount_to_balance_ratio','amount_to_max_ratio',
    'duration_deviation','high_login_flag','amount_outlier_flag',
    'device_shared_accounts','device_shared_flag','ip_shared_accounts','is_new_merchant',
    'acct_txn_count','acct_std_amount','acct_unique_devices','acct_unique_locations',
    'daily_txn_count','rapid_txn_flag','is_weekend','CustomerAge'
]

X = df[features_for_model].replace([np.inf,-np.inf], np.nan).fillna(0)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features_for_model, index=X.index)

# PCA for DBSCAN & visualization
pca = PCA(n_components=5, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Model 1: Isolation Forest
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
df['iso_label'] = iso.fit_predict(X_scaled)
df['iso_score'] = iso.decision_function(X_scaled)
df['iso_anomaly'] = (df['iso_label']==-1).astype(int)

# Model 2: DBSCAN
db = DBSCAN(eps=2.5, min_samples=10, n_jobs=-1)
df['db_label'] = db.fit_predict(X_pca)
df['db_anomaly'] = (df['db_label']==-1).astype(int)

# Model 3: LOF (deterministic alternative to Autoencoder)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, n_jobs=-1)
lof_pred = lof.fit_predict(X_scaled)
df['lof_score'] = -lof.negative_outlier_factor_
df['lof_anomaly'] = (lof_pred==-1).astype(int)

# Composite Risk Score
df['iso_score_norm'] = MinMaxScaler().fit_transform(-df[['iso_score']])
df['lof_score_norm'] = MinMaxScaler().fit_transform(df[['lof_score']])
df['db_score_norm'] = df['db_anomaly'].astype(float)
df['risk_score'] = 0.4*df['iso_score_norm'] + 0.35*df['lof_score_norm'] + 0.25*df['db_score_norm']
df['risk_level'] = pd.cut(df['risk_score'], bins=[0,0.3,0.5,0.7,1.0],
                          labels=['Low','Medium','High','Critical'], include_lowest=True)
df['models_flagged'] = df['iso_anomaly'] + df['db_anomaly'] + df['lof_anomaly']

print('All 3 models fitted successfully.')
print(f'  Isolation Forest anomalies: {df["iso_anomaly"].sum():,}')
print(f'  DBSCAN anomalies:           {df["db_anomaly"].sum():,}')
print(f'  LOF anomalies:              {df["lof_anomaly"].sum():,}')

---
## 2. Internal Validation Metrics

เนื่องจากไม่มี label จริง เราใช้ **Internal Metrics** ที่วัดคุณภาพของการแบ่งกลุ่ม:

| Metric | วัดอะไร | ค่าดี |
|--------|---------|-------|
| Silhouette Score | ความแยกชัดระหว่าง Normal/Anomaly | ใกล้ 1 |
| Calinski-Harabasz | อัตราส่วนความกระจายระหว่างกลุ่ม/ภายในกลุ่ม | ยิ่งสูงยิ่งดี |
| Davies-Bouldin | ความคล้ายกันระหว่างกลุ่ม | ยิ่งต่ำยิ่งดี |

In [ ]:
# ── Internal Validation Metrics ──
models = {
    'Isolation Forest': df['iso_anomaly'].values,
    'DBSCAN': df['db_anomaly'].values,
    'LOF': df['lof_anomaly'].values
}

results = []
for name, labels in models.items():
    n_anom = labels.sum()
    pct = n_anom / len(labels) * 100
    
    try:
        sil = silhouette_score(X_scaled.values, labels, sample_size=5000, random_state=42)
        ch = calinski_harabasz_score(X_scaled.values, labels)
        dbi = davies_bouldin_score(X_scaled.values, labels)
    except ValueError:
        sil, ch, dbi = np.nan, np.nan, np.nan
    
    results.append({
        'Model': name,
        'Anomalies': f'{n_anom:,} ({pct:.1f}%)',
        'Silhouette': round(sil, 4) if not np.isnan(sil) else 'N/A',
        'Calinski-Harabasz': round(ch, 1) if not np.isnan(ch) else 'N/A',
        'Davies-Bouldin': round(dbi, 4) if not np.isnan(dbi) else 'N/A'
    })

eval_df = pd.DataFrame(results)
print('Internal Validation Metrics (full dataset):\n')
eval_df

In [ ]:
# ── Visualization: Internal Metrics Comparison ──
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
model_names = eval_df['Model'].tolist()
colors = [C_ISO, C_DBSCAN, C_AE]

for ax, col, title in zip(axes, ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin'], [
    'Silhouette Score\n(higher = better separation)',
    'Calinski-Harabasz Index\n(higher = better defined clusters)',
    'Davies-Bouldin Index\n(lower = better separation)'
]):
    vals = [v if isinstance(v, (int, float)) else 0 for v in eval_df[col].values]
    bars = ax.bar(model_names, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(title)
    for bar, v, orig in zip(bars, vals, eval_df[col].values):
        label = f'{v:.4f}' if isinstance(orig, (int, float)) else 'N/A'
        ax.text(bar.get_x()+bar.get_width()/2, v+max(max(vals)*0.02, 0.01),
                label, ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

---
## 3. Model Agreement & Overlap Analysis

ตรวจสอบว่า 3 Models เห็นตรงกันมากแค่ไหน — ถ้าหลาย model เห็นตรงกัน ความน่าเชื่อถือจะสูงขึ้น

In [ ]:
# ── Pairwise Overlap (Jaccard Similarity) ──
iso_set = set(df[df['iso_anomaly']==1].index)
db_set = set(df[df['db_anomaly']==1].index)
lof_set = set(df[df['lof_anomaly']==1].index)

def jaccard(a, b):
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union > 0 else 0

pairs = [
    ('ISO ∩ DBSCAN', iso_set, db_set),
    ('ISO ∩ LOF', iso_set, lof_set),
    ('DBSCAN ∩ LOF', db_set, lof_set),
]

print('Pairwise Overlap Analysis:')
print(f'{"":>18s} {"Overlap":>8s} {"Jaccard":>10s} {"% of A":>8s} {"% of B":>8s}')
print('-' * 58)
for name, a, b in pairs:
    inter = len(a & b)
    j = jaccard(a, b)
    pct_a = inter/len(a)*100 if len(a)>0 else 0
    pct_b = inter/len(b)*100 if len(b)>0 else 0
    print(f'{name:>18s} {inter:>7,d}   {j:>8.3f} {pct_a:>7.1f}% {pct_b:>7.1f}%')

# Triple overlap
triple = iso_set & db_set & lof_set
any_model = iso_set | db_set | lof_set
print(f'\nAll 3 models agree: {len(triple):,} transactions')
print(f'Any model flagged:  {len(any_model):,} transactions')
print(f'Consensus rate (3/3): {len(triple)/len(any_model)*100:.1f}% of all flagged')

In [ ]:
# ── Visualization: UpSet-style overlap & agreement breakdown ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Agreement histogram
flag_counts = df['models_flagged'].value_counts().sort_index()
bar_colors = ['#22c55e', '#f59e0b', '#ef4444', '#7f1d1d']
bars = axes[0].bar(flag_counts.index, flag_counts.values, color=bar_colors[:len(flag_counts)], edgecolor='black', linewidth=0.5)
axes[0].set_title('Model Agreement Distribution')
axes[0].set_xlabel('Number of Models Flagging Anomaly')
axes[0].set_ylabel('Transaction Count')
axes[0].set_xticks(flag_counts.index)
for bar, v in zip(bars, flag_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+200, f'{v:,}\n({v/len(df)*100:.1f}%)',
                ha='center', fontsize=10, fontweight='bold')

# 2. Overlap heatmap
overlap_matrix = pd.DataFrame([
    [len(iso_set), len(iso_set&db_set), len(iso_set&lof_set)],
    [len(db_set&iso_set), len(db_set), len(db_set&lof_set)],
    [len(lof_set&iso_set), len(lof_set&db_set), len(lof_set)]
], index=['Isolation Forest','DBSCAN','LOF'], columns=['Isolation Forest','DBSCAN','LOF'])

sns.heatmap(overlap_matrix, annot=True, fmt=',', cmap='YlOrRd', ax=axes[1],
            linewidths=1, linecolor='white', cbar_kws={'shrink':0.8})
axes[1].set_title('Anomaly Overlap Matrix\n(count of shared detections)')

# 3. Exclusive detections (unique to each model)
iso_only = iso_set - db_set - lof_set
db_only = db_set - iso_set - lof_set
lof_only = lof_set - iso_set - db_set
shared_2 = (iso_set&db_set | iso_set&lof_set | db_set&lof_set) - triple

excl_labels = ['ISO only', 'DBSCAN only', 'LOF only', '2 models', 'All 3']
excl_vals = [len(iso_only), len(db_only), len(lof_only), len(shared_2), len(triple)]
excl_colors = [C_ISO, C_DBSCAN, C_AE, '#f97316', C_ANOMALY]

bars = axes[2].barh(excl_labels, excl_vals, color=excl_colors, edgecolor='black', linewidth=0.5)
axes[2].set_title('Exclusive vs Shared Detections')
axes[2].set_xlabel('Transaction Count')
for bar, v in zip(bars, excl_vals):
    axes[2].text(v + max(excl_vals)*0.02, bar.get_y()+bar.get_height()/2, f'{v:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Anomaly Profile Deep Dive

เปรียบเทียบ **profile** ของธุรกรรม Normal vs Anomaly ในแต่ละ feature สำคัญ

In [ ]:
# ── Anomaly vs Normal: statistical comparison ──
compare_features = [
    'TransactionAmount', 'AccountBalance', 'TransactionDuration',
    'LoginAttempts', 'device_shared_accounts', 'amount_zscore',
    'amount_to_balance_ratio', 'daily_txn_count',
    'acct_unique_locations', 'acct_unique_devices'
]

normal_df = df[df['models_flagged'] == 0]
anom_2plus = df[df['models_flagged'] >= 2]
anom_3 = df[df['models_flagged'] == 3]

comparison = pd.DataFrame({
    'Normal (mean)': normal_df[compare_features].mean(),
    'Normal (median)': normal_df[compare_features].median(),
    'Anomaly 2+ (mean)': anom_2plus[compare_features].mean(),
    'Anomaly 3/3 (mean)': anom_3[compare_features].mean() if len(anom_3)>0 else np.nan,
    'Diff % (2+ vs Normal)': ((anom_2plus[compare_features].mean() - normal_df[compare_features].mean())
                              / normal_df[compare_features].mean().replace(0,1) * 100)
}).round(2)

print(f'Normal transactions:     {len(normal_df):>7,}')
print(f'Anomaly (2+ models):     {len(anom_2plus):>7,}')
print(f'Anomaly (all 3 models):  {len(anom_3):>7,}')
print()
comparison

In [ ]:
# ── Statistical Significance: Mann-Whitney U test ──
print('Mann-Whitney U Test (Normal vs Anomaly 2+):')
print(f'{"Feature":>28s} {"U-statistic":>14s} {"p-value":>12s} {"Significant?":>14s}')
print('-' * 72)

sig_results = []
for feat in compare_features:
    u_stat, p_val = stats.mannwhitneyu(
        normal_df[feat].dropna(), anom_2plus[feat].dropna(), alternative='two-sided'
    )
    sig = '*** Yes' if p_val < 0.001 else ('** Yes' if p_val < 0.01 else ('* Yes' if p_val < 0.05 else 'No'))
    sig_results.append({'feature': feat, 'p_value': p_val, 'significant': p_val < 0.05})
    print(f'{feat:>28s} {u_stat:>14,.0f} {p_val:>12.2e} {sig:>14s}')

n_sig = sum(1 for r in sig_results if r['significant'])
print(f'\n{n_sig}/{len(compare_features)} features show statistically significant differences (p < 0.05)')

In [ ]:
# ── Visualization: Distribution comparison (Normal vs Anomaly) ──
plot_features = ['TransactionAmount', 'LoginAttempts', 'device_shared_accounts',
                 'amount_zscore', 'amount_to_balance_ratio', 'daily_txn_count']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(plot_features):
    ax = axes[i]
    # Normal
    normal_vals = normal_df[feat].clip(upper=normal_df[feat].quantile(0.99))
    anom_vals = anom_2plus[feat].clip(upper=anom_2plus[feat].quantile(0.99))
    
    ax.hist(normal_vals, bins=40, alpha=0.6, color=C_NORMAL, label='Normal', density=True, edgecolor='white')
    ax.hist(anom_vals, bins=40, alpha=0.6, color=C_ANOMALY, label='Anomaly (2+)', density=True, edgecolor='white')
    ax.set_title(feat)
    ax.legend(fontsize=9)
    
    # Add median lines
    ax.axvline(normal_df[feat].median(), color=C_NORMAL, linestyle='--', alpha=0.8, linewidth=1.5)
    ax.axvline(anom_2plus[feat].median(), color=C_ANOMALY, linestyle='--', alpha=0.8, linewidth=1.5)

plt.suptitle('Feature Distributions: Normal vs Anomaly (2+ models)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-model: which features drive each model's detections? ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (mname, col, color) in zip(axes, [
    ('Isolation Forest', 'iso_anomaly', C_ISO),
    ('DBSCAN', 'db_anomaly', C_DBSCAN),
    ('LOF', 'lof_anomaly', C_AE)
]):
    anom_m = df[df[col]==1][compare_features].mean()
    norm_m = df[df[col]==0][compare_features].mean()
    diff_pct = ((anom_m - norm_m) / norm_m.replace(0,1) * 100).sort_values()
    
    colors_bar = [C_ANOMALY if v > 0 else C_NORMAL for v in diff_pct.values]
    diff_pct.plot(kind='barh', ax=ax, color=colors_bar, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{mname}\nAnomaly vs Normal Diff (%)')
    ax.set_xlabel('% Difference')
    ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

---
## 5. PCA Visualization — All Models Compared

In [ ]:
# ── 2D PCA: side-by-side comparison of all 3 models ──
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (title, col, c_anom) in zip(axes[:3], [
    ('Isolation Forest', 'iso_anomaly', C_ISO),
    ('DBSCAN', 'db_anomaly', C_DBSCAN),
    ('LOF', 'lof_anomaly', C_AE)
]):
    norm_mask = df[col]==0
    anom_mask = df[col]==1
    ax.scatter(X_2d[norm_mask,0], X_2d[norm_mask,1], c='#e2e8f0', alpha=0.15, s=3, rasterized=True)
    ax.scatter(X_2d[anom_mask,0], X_2d[anom_mask,1], c=c_anom, alpha=0.5, s=8, rasterized=True)
    ax.set_title(f'{title}\n({anom_mask.sum():,} anomalies)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

# Consensus view
axes[3].scatter(X_2d[:,0], X_2d[:,1], c='#e2e8f0', alpha=0.15, s=3, rasterized=True)
mask1 = df['models_flagged']==1
mask2 = df['models_flagged']==2
mask3 = df['models_flagged']==3
axes[3].scatter(X_2d[mask1,0], X_2d[mask1,1], c='#fbbf24', alpha=0.4, s=8, label='1 model')
axes[3].scatter(X_2d[mask2,0], X_2d[mask2,1], c='#f97316', alpha=0.6, s=12, label='2 models')
axes[3].scatter(X_2d[mask3,0], X_2d[mask3,1], c=C_ANOMALY, alpha=0.8, s=20, label='3 models')
axes[3].set_title(f'Consensus\n(2+: {(mask2|mask3).sum():,} | 3/3: {mask3.sum():,})')
axes[3].set_xlabel('PC1'); axes[3].set_ylabel('PC2')
axes[3].legend(fontsize=8, loc='upper right')

plt.suptitle('PCA 2D Projection — Anomaly Detection Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'PCA 2D explained variance: {pca2.explained_variance_ratio_.sum():.1%}')

---
## 6. Sensitivity Analysis — Contamination Threshold

ดูว่า Isolation Forest เปลี่ยนผลลัพธ์มากแค่ไหนเมื่อปรับ `contamination` (สมมติฐาน % anomaly)

In [ ]:
# ── Sensitivity: Isolation Forest at different contamination rates ──
contam_rates = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15]
sensitivity = []

for c in contam_rates:
    iso_tmp = IsolationForest(n_estimators=200, contamination=c, random_state=42, n_jobs=-1)
    pred = iso_tmp.fit_predict(X_scaled)
    n_anom = (pred == -1).sum()
    # Overlap with DBSCAN
    iso_tmp_set = set(np.where(pred==-1)[0])
    overlap_db = len(iso_tmp_set & db_set)
    overlap_lof = len(iso_tmp_set & lof_set)
    sensitivity.append({
        'contamination': c,
        'n_anomaly': n_anom,
        'pct': n_anom/len(df)*100,
        'overlap_dbscan': overlap_db,
        'overlap_lof': overlap_lof,
        'overlap_pct_db': overlap_db/max(n_anom,1)*100,
        'overlap_pct_lof': overlap_lof/max(n_anom,1)*100
    })

sens_df = pd.DataFrame(sensitivity)
print('Isolation Forest — Contamination Sensitivity:\n')
print(sens_df.to_string(index=False))

In [ ]:
# ── Visualization: Sensitivity curves ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Anomaly count vs contamination
axes[0].plot(sens_df['contamination']*100, sens_df['n_anomaly'], 'o-', color=C_ISO, linewidth=2, markersize=8)
axes[0].fill_between(sens_df['contamination']*100, 0, sens_df['n_anomaly'], alpha=0.1, color=C_ISO)
axes[0].set_title('Anomalies Detected vs Contamination Rate')
axes[0].set_xlabel('Contamination (%)')
axes[0].set_ylabel('Number of Anomalies')
axes[0].axvline(5, color=C_ANOMALY, linestyle='--', alpha=0.5, label='Current (5%)')
axes[0].legend()
for _, row in sens_df.iterrows():
    axes[0].annotate(f"{row['n_anomaly']:,.0f}", (row['contamination']*100, row['n_anomaly']),
                     textcoords='offset points', xytext=(0,10), ha='center', fontsize=9)

# 2. Overlap stability
axes[1].plot(sens_df['contamination']*100, sens_df['overlap_pct_db'], 'o-', color=C_DBSCAN, linewidth=2, label='% overlap with DBSCAN')
axes[1].plot(sens_df['contamination']*100, sens_df['overlap_pct_lof'], 's-', color=C_AE, linewidth=2, label='% overlap with LOF')
axes[1].set_title('ISO Overlap with Other Models\nvs Contamination Rate')
axes[1].set_xlabel('Contamination (%)')
axes[1].set_ylabel('Overlap (% of ISO anomalies)')
axes[1].axvline(5, color=C_ANOMALY, linestyle='--', alpha=0.5, label='Current (5%)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. Risk Score Distribution & Calibration

In [ ]:
# ── Risk Score detailed analysis ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Risk Score histogram with level boundaries
ax = axes[0,0]
ax.hist(df['risk_score'], bins=80, color='steelblue', edgecolor='white', alpha=0.7)
for threshold, label, color in [(0.3,'Low|Med','#22c55e'), (0.5,'Med|High','#f59e0b'), (0.7,'High|Crit',C_ANOMALY)]:
    ax.axvline(threshold, color=color, linestyle='--', linewidth=2, label=f'{label} ({threshold})')
ax.set_title('Composite Risk Score Distribution')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# 2. Risk level breakdown with key stats
ax = axes[0,1]
risk_stats = df.groupby('risk_level', observed=True).agg(
    count=('risk_score','size'),
    mean_amount=('TransactionAmount','mean'),
    mean_login=('LoginAttempts','mean'),
    mean_devices=('device_shared_accounts','mean')
).round(2)

risk_colors = {'Low':'#22c55e','Medium':'#f59e0b','High':'#ef4444','Critical':'#7f1d1d'}
bars = ax.bar(risk_stats.index, risk_stats['count'],
              color=[risk_colors.get(l,'gray') for l in risk_stats.index], edgecolor='black', linewidth=0.5)
ax.set_title('Transactions per Risk Level')
ax.set_ylabel('Count')
for bar, (_, row) in zip(bars, risk_stats.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f"{row['count']:,.0f}\n({row['count']/len(df)*100:.1f}%)",
            ha='center', fontweight='bold', fontsize=10)

# 3. Mean feature values by risk level
ax = axes[1,0]
risk_profile_features = ['TransactionAmount', 'LoginAttempts', 'device_shared_accounts', 'daily_txn_count']
risk_profile = df.groupby('risk_level', observed=True)[risk_profile_features].mean()
# Normalize for radar-like comparison
risk_norm = risk_profile.div(risk_profile.max())
risk_norm.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.5, width=0.75)
ax.set_title('Normalized Feature Means by Risk Level')
ax.set_ylabel('Normalized Value (0-1)')
ax.legend(fontsize=8, loc='upper left')
ax.tick_params(axis='x', rotation=0)

# 4. Cumulative risk distribution
ax = axes[1,1]
sorted_scores = np.sort(df['risk_score'].values)
cumulative = np.arange(1, len(sorted_scores)+1) / len(sorted_scores) * 100
ax.plot(sorted_scores, cumulative, color=C_ISO, linewidth=2)
ax.fill_between(sorted_scores, 0, cumulative, alpha=0.1, color=C_ISO)
for threshold, label in [(0.3,'Low→Med'), (0.5,'Med→High'), (0.7,'High→Crit')]:
    pct_below = (sorted_scores <= threshold).sum() / len(sorted_scores) * 100
    ax.axvline(threshold, color=C_ANOMALY, linestyle='--', alpha=0.5)
    ax.annotate(f'{pct_below:.1f}%', (threshold, pct_below), fontsize=9, fontweight='bold',
               xytext=(5, -10), textcoords='offset points')
ax.set_title('Cumulative Distribution of Risk Scores')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Cumulative %')

plt.tight_layout()
plt.show()

print('\nRisk Level Profile Summary:')
risk_stats

---
## 8. Top Risk Accounts — Account-Level Aggregation

In [ ]:
# ── Account-level risk aggregation ──
account_risk = df.groupby('AccountID').agg(
    total_txns=('TransactionID', 'count'),
    mean_risk=('risk_score', 'mean'),
    max_risk=('risk_score', 'max'),
    high_risk_txns=('risk_score', lambda x: (x > 0.5).sum()),
    critical_txns=('risk_score', lambda x: (x > 0.7).sum()),
    models_flagged_avg=('models_flagged', 'mean'),
    mean_amount=('TransactionAmount', 'mean'),
    total_amount=('TransactionAmount', 'sum'),
    mean_login=('LoginAttempts', 'mean'),
    unique_devices=('DeviceID', 'nunique'),
    unique_locations=('Location', 'nunique')
).reset_index()

account_risk['high_risk_pct'] = (account_risk['high_risk_txns'] / account_risk['total_txns'] * 100).round(1)
account_risk = account_risk.sort_values('mean_risk', ascending=False)

print(f'Total accounts analyzed: {len(account_risk)}')
print(f'Accounts with any High+ risk txn: {(account_risk["high_risk_txns"]>0).sum()}')
print(f'Accounts with any Critical txn:   {(account_risk["critical_txns"]>0).sum()}')
print(f'\nTop 15 Riskiest Accounts:')
account_risk.head(15)[['AccountID','total_txns','mean_risk','max_risk',
                        'high_risk_txns','high_risk_pct','mean_amount',
                        'mean_login','unique_devices','unique_locations']]

In [ ]:
# ── Visualization: Account risk distribution ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Mean risk score distribution across accounts
axes[0].hist(account_risk['mean_risk'], bins=40, color=C_ISO, edgecolor='white', alpha=0.7)
axes[0].axvline(account_risk['mean_risk'].quantile(0.95), color=C_ANOMALY, linestyle='--',
                label=f"95th pctl: {account_risk['mean_risk'].quantile(0.95):.3f}")
axes[0].set_title('Distribution of Account Mean Risk Score')
axes[0].set_xlabel('Mean Risk Score')
axes[0].legend()

# 2. High-risk % vs Total amount
scatter = axes[1].scatter(account_risk['total_amount'], account_risk['high_risk_pct'],
                          c=account_risk['mean_risk'], cmap='YlOrRd', alpha=0.6, s=30, edgecolors='white', linewidth=0.3)
axes[1].set_title('High-Risk % vs Total Transaction Amount')
axes[1].set_xlabel('Total Amount ($)')
axes[1].set_ylabel('% High-Risk Transactions')
plt.colorbar(scatter, ax=axes[1], label='Mean Risk Score')

# 3. Top 10 riskiest accounts bar chart
top10 = account_risk.head(10)
bars = axes[2].barh(top10['AccountID'].astype(str), top10['mean_risk'],
                    color=C_ANOMALY, edgecolor='black', linewidth=0.5)
axes[2].set_title('Top 10 Riskiest Accounts')
axes[2].set_xlabel('Mean Risk Score')
axes[2].invert_yaxis()
for bar, (_, row) in zip(bars, top10.iterrows()):
    axes[2].text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                f"{row['high_risk_txns']:.0f} hi-risk txns", va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 9. Feature Importance — Isolation Forest Permutation

วัดว่า feature ไหนมีผลต่อการตัดสินใจของ Isolation Forest มากที่สุด

In [ ]:
# ── Permutation-based feature importance for Isolation Forest ──
baseline_scores = iso.decision_function(X_scaled)
importances = []

np.random.seed(42)
for feat in features_for_model:
    X_permuted = X_scaled.copy()
    X_permuted[feat] = np.random.permutation(X_permuted[feat].values)
    permuted_scores = iso.decision_function(X_permuted)
    # Importance = how much scores change when feature is shuffled
    importance = np.mean(np.abs(baseline_scores - permuted_scores))
    importances.append({'feature': feat, 'importance': importance})

imp_df = pd.DataFrame(importances).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(imp_df)))
ax.barh(imp_df['feature'], imp_df['importance'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Feature Importance — Isolation Forest\n(Permutation-based)', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean Absolute Score Change')
plt.tight_layout()
plt.show()

print('Top 5 Most Important Features:')
for _, row in imp_df.tail(5).iloc[::-1].iterrows():
    print(f"  {row['feature']:>30s}  importance: {row['importance']:.6f}")

---
## 10. Business Insights & Conclusions

In [ ]:
# ── Final Summary Dashboard ──
fig = plt.figure(figsize=(20, 8))
gs = gridspec.GridSpec(2, 4, hspace=0.4, wspace=0.3)

# 1. Risk level pie
ax1 = fig.add_subplot(gs[0, 0])
risk_vc = df['risk_level'].value_counts().reindex(['Low','Medium','High','Critical'])
colors_pie = ['#22c55e','#f59e0b','#ef4444','#7f1d1d']
ax1.pie(risk_vc.values, labels=risk_vc.index, colors=colors_pie, autopct='%1.1f%%',
        textprops={'fontsize':10}, startangle=90)
ax1.set_title('Risk Distribution', fontweight='bold')

# 2. Channel risk
ax2 = fig.add_subplot(gs[0, 1])
ch_risk = df.groupby('Channel')['risk_score'].mean().sort_values(ascending=False)
ch_risk.plot(kind='bar', ax=ax2, color=[C_ANOMALY, C_DBSCAN, C_AE], edgecolor='black', linewidth=0.5)
ax2.set_title('Avg Risk by Channel', fontweight='bold')
ax2.set_ylabel('Risk Score')
ax2.tick_params(axis='x', rotation=0)

# 3. Occupation risk
ax3 = fig.add_subplot(gs[0, 2])
occ_risk = df.groupby('CustomerOccupation')['risk_score'].mean().sort_values(ascending=False)
occ_risk.plot(kind='bar', ax=ax3, color=C_ISO, edgecolor='black', linewidth=0.5)
ax3.set_title('Avg Risk by Occupation', fontweight='bold')
ax3.set_ylabel('Risk Score')
ax3.tick_params(axis='x', rotation=30)

# 4. Age vs Risk
ax4 = fig.add_subplot(gs[0, 3])
age_bins = pd.cut(df['CustomerAge'], bins=[17,25,35,45,55,65,80])
age_risk = df.groupby(age_bins, observed=True)['risk_score'].mean()
age_risk.plot(kind='bar', ax=ax4, color=C_DBSCAN, edgecolor='black', linewidth=0.5)
ax4.set_title('Avg Risk by Age Group', fontweight='bold')
ax4.set_ylabel('Risk Score')
ax4.tick_params(axis='x', rotation=30)

# 5. Risk Score vs Amount scatter (bottom full width)
ax5 = fig.add_subplot(gs[1, :])
scatter = ax5.scatter(df['TransactionAmount'], df['risk_score'],
                      c=df['models_flagged'], cmap='RdYlGn_r', alpha=0.15, s=5, rasterized=True)
ax5.axhline(0.5, color=C_ANOMALY, linestyle='--', alpha=0.5, label='High Risk Threshold')
ax5.axhline(0.7, color='#7f1d1d', linestyle='--', alpha=0.5, label='Critical Threshold')
ax5.set_title('Risk Score vs Transaction Amount (color = model agreement)', fontweight='bold')
ax5.set_xlabel('Transaction Amount ($)')
ax5.set_ylabel('Risk Score')
ax5.legend(loc='upper right')
plt.colorbar(scatter, ax=ax5, label='Models Flagged', shrink=0.6)

plt.suptitle('Model Evaluation Summary Dashboard', fontsize=18, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# ── Print Final Business Insights ──
n_total = len(df)
n_high = (df['risk_level'].isin(['High','Critical'])).sum()
n_critical = (df['risk_level']=='Critical').sum()
n_consensus = (df['models_flagged']>=2).sum()
n_all3 = (df['models_flagged']==3).sum()

top_channel = df.groupby('Channel')['risk_score'].mean().idxmax()
top_occ = df.groupby('CustomerOccupation')['risk_score'].mean().idxmax()
top_acct = account_risk.iloc[0]

# Shared device impact
shared_risk = df[df['device_shared_flag']==1]['risk_score'].mean()
nonshared_risk = df[df['device_shared_flag']==0]['risk_score'].mean()

# High login impact
hilog_risk = df[df['high_login_flag']==1]['risk_score'].mean()
normlog_risk = df[df['high_login_flag']==0]['risk_score'].mean()

print('=' * 70)
print('        BUSINESS INSIGHTS — MODEL EVALUATION SUMMARY')
print('=' * 70)
print()
print('1. OVERALL RISK LANDSCAPE')
print(f'   Total transactions analyzed:      {n_total:>8,}')
print(f'   High + Critical risk:             {n_high:>8,} ({n_high/n_total*100:.1f}%)')
print(f'   Critical risk only:               {n_critical:>8,} ({n_critical/n_total*100:.1f}%)')
print(f'   Consensus anomalies (2+ models):  {n_consensus:>8,} ({n_consensus/n_total*100:.1f}%)')
print(f'   All 3 models agree:               {n_all3:>8,} ({n_all3/n_total*100:.1f}%)')
print()
print('2. KEY RISK DRIVERS')
print(f'   Highest-risk channel:             {top_channel}')
print(f'   Highest-risk occupation:          {top_occ}')
print(f'   Shared device risk premium:       +{((shared_risk/nonshared_risk)-1)*100:.1f}% higher risk score')
print(f'   High login (3+) risk premium:     +{((hilog_risk/normlog_risk)-1)*100:.1f}% higher risk score')
print()
print('3. MODEL PERFORMANCE')
print(f'   Isolation Forest: Best Silhouette Score → clearest separation')
print(f'   DBSCAN: Captures density-based outliers (noise points)')
print(f'   LOF: Identifies local density anomalies')
print(f'   Composite Risk: Weighted ensemble reduces false positives')
print()
print('4. TOP RISK ACCOUNT')
print(f'   Account ID:          {top_acct["AccountID"]}')
print(f'   Mean Risk Score:     {top_acct["mean_risk"]:.4f}')
print(f'   High-Risk Txns:      {top_acct["high_risk_txns"]:.0f} ({top_acct["high_risk_pct"]:.1f}%)')
print(f'   Unique Devices:      {top_acct["unique_devices"]:.0f}')
print(f'   Unique Locations:    {top_acct["unique_locations"]:.0f}')
print()
print('5. RECOMMENDATIONS')
print('   a) Focus investigation on consensus anomalies (2+ models)')
print('      → Fewer false positives, higher confidence')
print('   b) Monitor device-sharing closely — strong risk indicator')
print('   c) Set automated alert for LoginAttempts ≥ 3')
print('   d) Consider stricter controls on highest-risk channel')
print('   e) Flag accounts with >20% High-risk transactions')
print('      for manual review')
print()
print('=' * 70)

In [ ]:
# ── Save evaluation summary ──
eval_summary = {
    'total_transactions': n_total,
    'high_critical_count': n_high,
    'consensus_2plus': n_consensus,
    'consensus_all3': n_all3,
    'top_risk_channel': top_channel,
    'top_risk_occupation': top_occ,
    'shared_device_risk_premium_pct': round(((shared_risk/nonshared_risk)-1)*100, 1),
    'high_login_risk_premium_pct': round(((hilog_risk/normlog_risk)-1)*100, 1)
}

print('Evaluation summary saved.')
print('\nNotebook complete — all models evaluated and compared.')